# 04 NLP Portuguese Review Analysis

Notebook ini menganalisis teks review pelanggan berbahasa Brazilian Portuguese untuk menjawab pertanyaan: **apa yang dikeluhkan pelanggan dengan low rating?**

Analisis ini bersifat deskriptif/root-cause oriented. Review text tidak digunakan sebagai fitur prediksi ML low-rating karena teks muncul setelah review dikirim dan akan menyebabkan leakage.

## 1. Imports and Path Setup

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from scripts.nlp_portuguese_review_analysis import (
    load_reviews,
    prepare_cleaned_reviews,
    tokenize_normalized_text,
    load_stopwords,
    maybe_get_stemmer,
    tag_review_themes,
    summarize_themes,
    build_top_terms,
    build_nmf_topics,
    build_lda_topics,
    build_audit,
    write_outputs,
    THEME_PATTERNS,
    OUTPUT_DIR,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 100)

## 2. Load Dataset and Initial Audit

In [2]:
reviews = load_reviews()
reviews.shape

(99224, 7)

In [3]:
reviews[["review_id", "order_id", "review_score", "review_comment_title", "review_comment_message"]].sample(5, random_state=42)

,review_id,order_id,review_score,review_comment_title,review_comment_message
90252,406e32984dd5273582105460a79571af,e00ed9d20c3479f9f0e9727ca9d60946,5,NaN,NaN
24436,3bca8d9922bed47eb96f23d121945290,3b5351f5f99b46339212291661a9d226,5,NaN,Cumpriu o acordado!
11313,65895b807ac5dfe062c82400b3f210b8,f395e98fb5c1c6ce1306e80de2fe125b,4,NaN,NaN
75442,2a6faa65a6e893105c60b1018d40e14a,57899333b5e286632bd2599d3f7864ce,5,NaN,NaN
7217,a738aa683a09dc5979abc7d9c2cc8029,0cc76fbe09687fda664178e9fc6c404f,5,NaN,NaN


In [4]:
initial_audit = pd.Series({
    "total_reviews": len(reviews),
    "reviews_with_title": reviews["review_comment_title"].fillna("").str.strip().ne("").sum(),
    "reviews_with_message": reviews["review_comment_message"].fillna("").str.strip().ne("").sum(),
    "low_rating_reviews": reviews["review_score"].le(2).sum(),
})
initial_audit

total_reviews           99224
reviews_with_title      11566
reviews_with_message    40950
low_rating_reviews      14575
dtype: int64

## 3. Combine Review Title and Message

In [5]:
cleaned = prepare_cleaned_reviews(reviews)
cleaned[["review_id", "review_score", "review_text_raw"]].head(10)

,review_id,review_score,review_text_raw
0,7bc2406110b926393aa56f80a40eba40,4,
1,80e641a11e56f04c1ad469d5645fdfde,5,
2,228ce5500dc1d8e020d8d1322874b6f0,5,
3,e64fb393e7b32834bb789ff8bb30750e,5,Recebi bem antes do prazo estipulado.
4,f7c4243c7fe1938f181bec41a392bdeb,5,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa
5,15197aa66ff4d0650b5434f1b46cda19,1,
6,07f9bee5d1b850860defd761afa7ff16,5,
7,7c6400515c67679fbee952a7525281ef,5,
8,a3f6f7f6f433de0aefbb97da197c554c,5,
9,8670d52e15e00043ae7de4c01cc2fe06,4,recomendo aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar co...


## 4. Portuguese Text Cleaning

`clean_text` mempertahankan aksen Portugis agar tetap readable. `normalized_text` menghapus aksen untuk matching keyword.

In [6]:
cleaned[["review_text_raw", "clean_text", "normalized_text", "token_count", "has_text", "is_low_rating"]].sample(10, random_state=7)

,review_text_raw,clean_text,normalized_text,token_count,has_text,is_low_rating
53576,Ótimo recomendo é ótimo,ótimo recomendo é ótimo,otimo recomendo e otimo,4,True,False
60091,,,,0,False,False
27165,,,,0,False,False
2273,,,,0,False,False
57791,,,,0,False,False
62451,,,,0,False,False
44810,,,,0,False,False
74930,Ótimo produto...,ótimo produto.,otimo produto,2,True,False
42016,"comprei duas mangueiras e so recebi uma, na nota fiscal conta duas e no pacote veio so uma. Quero uma solução para este problema. Desde ...","comprei duas mangueiras e so recebi uma, na nota fiscal conta duas e no pacote veio so uma. quero uma solução para este problema. desde ...",comprei duas mangueiras e so recebi uma na nota fiscal conta duas e no pacote veio so uma quero uma solucao para este problema desde ja ...,27,True,True
54917,"Chegou no prazo, tudo certo","chegou no prazo, tudo certo",chegou no prazo tudo certo,5,True,False


## 5. Phrase-Aware Preprocessing

`phrase_text` mengubah frasa keluhan penting menjadi token underscore seperti `nao_recebi`, `nao_chegou`, `produto_errado`, dan `fora_prazo`.

In [7]:
phrase_examples = cleaned[cleaned["phrase_text"].str.contains("nao_recebi|nao_chegou|produto_errado|fora_prazo", regex=True, na=False)]
phrase_examples[["review_score", "clean_text", "phrase_text"]].head(10)

,review_score,clean_text,phrase_text
19,1,não chegou meu produto péssimo,nao_chegou meu produto pessimo
32,1,"sempre compro pela internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. no stark o prazo máximo já se ...",sempre compro pela internet e a entrega ocorre antes do prazo combinado que acredito ser o prazo maximo no stark o prazo maximo ja se es...
68,1,"o produto não chegou no prazo estipulado e causou transtorno, pq programei a viagem de férias do meu filho, baseado no prazo. moro na ba...",o nao_chegou no prazo estipulado e causou transtorno pq programei a viagem de ferias do meu filho baseado no prazo moro na bahia e ele e...
104,3,não recebi o produto.,nao_recebi o produto
149,1,eu não recebi o produto e consta no sistema que eu recebi além de pagar caro do frete,eu nao_recebi o produto e consta no sistema que eu recebi alem de pagar caro do frete
169,1,fiz minha compra faz 30 dias e não recebi ainda meu produto. precisa melhorar nas entregas,fiz minha compra faz 30 dias e nao_recebi ainda meu produto precisa melhorar nas entregas
180,3,"comprei dois lustres pendentes, com a parceira targaryen e só me enviaram um lustre. abri reclamação, mas ainda não recebi resposta. agu...",comprei dois lustres pendentes com a parceira targaryen e so me enviaram um lustre abri reclamacao mas nao_recebi resposta aguardo soluc...
197,1,não recebi ainda aqui está descrevendo como entregue só que ate agora não recebi,nao_recebi ainda aqui esta descrevendo como entregue so que ate agora nao_recebi
241,1,o meu produto não chegou. segundo o rastreio está parado em ribeirão preto sp.,o meu nao_chegou segundo o rastreio esta parado em ribeirao preto sp
245,1,até agora não recebi o produto.,ate agora nao_recebi o produto


## 6. Library-Based Portuguese Stopwords and Custom Stopwords

Stopwords Portugis dari NLTK digunakan bila tersedia. Jika resource NLTK tidak tersedia, fallback lokal digunakan agar notebook tetap bisa berjalan offline. Negasi seperti `nao`, `nunca`, `nem`, dan `sem` dipertahankan.

In [8]:
all_normalized_tokens = [
    token
    for text in cleaned.loc[cleaned["has_text"], "normalized_text"]
    for token in tokenize_normalized_text(text)
]
matching_stopwords, topic_stopwords = load_stopwords(all_normalized_tokens)
stemmer = maybe_get_stemmer()

len(matching_stopwords), len(topic_stopwords), stemmer is not None

c:\Kuliah ITS Farhan\Semester 4\Admin_Lab\MCI\FP_MCI_Farhan\farhan_fp_mci_customer_experience\scripts\nlp_portuguese_review_analysis.py:337: UserWarning: RSLPStemmer unavailable; continuing without stemming.
  warnings.warn("RSLPStemmer unavailable; continuing without stemming.")


(202, 229, False)

In [9]:
protected_terms = ["nao", "nunca", "nem", "sem", "nao_recebi", "nao_chegou", "produto_errado", "fora_prazo", "nao_funciona", "sem_resposta"]
{term: term in matching_stopwords or term in topic_stopwords for term in protected_terms}

{'nao': True,
 'nunca': False,
 'nem': False,
 'sem': False,
 'nao_recebi': False,
 'nao_chegou': False,
 'produto_errado': False,
 'fora_prazo': False,
 'nao_funciona': False,
 'sem_resposta': False}

## 7. Preprocessing Sanity Check

In [10]:
cleaned.loc[cleaned["is_low_rating"] & cleaned["has_text"], ["review_score", "review_text_raw", "clean_text", "normalized_text", "phrase_text"]].head(10)

,review_score,review_text_raw,clean_text,normalized_text,phrase_text
16,2,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E ESSA COMPRA AGORA ME DECPCIONOU","gostaria de saber o que houve, sempre recebi e essa compra agora me decpcionou",gostaria de saber o que houve sempre recebi e essa compra agora me decpcionou,gostaria de saber o que houve sempre recebi e essa compra agora me decpcionou
19,1,Não chegou meu produto Péssimo,não chegou meu produto péssimo,nao chegou meu produto pessimo,nao_chegou meu produto pessimo
29,1,Não gostei ! Comprei gato por lebre,não gostei ! comprei gato por lebre,nao gostei comprei gato por lebre,nao gostei comprei gato por lebre
32,1,"Sempre compro pela Internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. No stark o prazo máximo já se ...","sempre compro pela internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. no stark o prazo máximo já se ...",sempre compro pela internet e a entrega ocorre antes do prazo combinado que acredito ser o prazo maximo no stark o prazo maximo ja se es...,sempre compro pela internet e a entrega ocorre antes do prazo combinado que acredito ser o prazo maximo no stark o prazo maximo ja se es...
39,1,Nada de chegar o meu pedido.,nada de chegar o meu pedido.,nada de chegar o meu pedido,nada de chegar o meu pedido
51,1,recebi somente 1 controle Midea Split ESTILO. Faltou Controle Remoto para Ar Condicionado Consul,recebi somente 1 controle midea split estilo. faltou controle remoto para ar condicionado consul,recebi somente 1 controle midea split estilo faltou controle remoto para ar condicionado consul,recebi somente 1 controle midea split estilo faltou controle remoto para ar condicionado consul
68,1,"O produto não chegou no prazo estipulado e causou transtorno, pq programei a viagem de férias do meu filho, baseado no prazo. Moro na Ba...","o produto não chegou no prazo estipulado e causou transtorno, pq programei a viagem de férias do meu filho, baseado no prazo. moro na ba...",o produto nao chegou no prazo estipulado e causou transtorno pq programei a viagem de ferias do meu filho baseado no prazo moro na bahia...,o nao_chegou no prazo estipulado e causou transtorno pq programei a viagem de ferias do meu filho baseado no prazo moro na bahia e ele e...
76,1,"Produto muito inferior, mal acabado.","produto muito inferior, mal acabado.",produto muito inferior mal acabado,produto muito inferior mal_acabado
89,1,Pedi reembolso e sem resposta até momento,pedi reembolso e sem resposta até momento,pedi reembolso e sem resposta ate momento,pedi_reembolso e sem_resposta ate momento
115,1,"Este foi o pedido Balde Com 128 Peças - Blocos De Montar 2 un - R$ 25,00 cada (NÃO FOI ENTREGUE) Vendido e entregue targaryen Tapete de ...","este foi o pedido balde com 128 peças - blocos de montar 2 un - r 25,00 cada não foi entregue vendido e entregue targaryen tapete de eva...",este foi o pedido balde com 128 pecas blocos de montar 2 un r 25 00 cada nao foi entregue vendido e entregue targaryen tapete de eva n l...,este foi o pedido balde com 128 pecas blocos de montar 2 un r 25 00 cada nao foi entregue vendido e entregue targaryen tapete de eva n l...


## 8. Rule-Based Multi-Label Complaint Theme Tagging

Satu review dapat memiliki lebih dari satu tema.

In [11]:
THEME_PATTERNS

{'theme_delivery_not_received': ['nao_recebi',
  'nao_chegou',
  'aguardando',
  'consta entregue',
  'consta_entregue',
  'marcado como entregue',
  'marcado_entregue'],
 'theme_delivery_delay': ['atraso',
  'atrasado',
  'atrasada',
  'atrasou',
  'demora',
  'demorou',
  'fora_prazo',
  'prazo',
  'rastreio',
  'transportadora',
  'correios'],
 'theme_product_wrong_incomplete': ['produto_errado',
  'produto_diferente',
  'produto_incompleto',
  'faltou',
  'faltando',
  'incompleto',
  'diferente',
  'tamanho_errado',
  'cor_errada'],
 'theme_product_defect_quality': ['defeito',
  'defeituoso',
  'defeituosa',
  'quebrado',
  'quebrada',
  'quebrados',
  'danificado',
  'danificada',
  'danificados',
  'qualidade_ruim',
  'pessimo',
  'ruim',
  'nao_funciona',
  'mal_acabado'],
 'theme_seller_service_refund': ['reembolso',
  'estorno',
  'devolucao',
  'pedi_reembolso',
  'pedi_estorno',
  'atendimento',
  'sem_resposta',
  'contato',
  'vendedor',
  'reclamei',
  'troca'],
 'theme_

In [12]:
theme_tags = tag_review_themes(cleaned)
theme_tags.head()

,review_id,order_id,review_score,theme_delivery_not_received,theme_delivery_delay,theme_product_wrong_incomplete,theme_product_defect_quality,theme_seller_service_refund,theme_positive_recommendation,theme_other,theme_count
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,False,False,False,False,False,False,True,1
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,False,False,False,False,False,False,True,1
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,False,False,False,False,False,False,True,1
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,False,True,False,False,False,False,False,1
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,False,False,False,False,False,False,True,1


## 9. Theme Summary

In [13]:
theme_summary = summarize_themes(theme_tags, cleaned)
theme_summary

,theme,review_count,percentage_of_low_rating_reviews,avg_review_score,sample_keywords
6,Other Complaint,4068,36.14,1.21,no matched theme
0,Delivery - Not Received,2844,25.27,1.12,"nao_recebi, nao_chegou, aguardando, consta entregue, consta_entregue, marcado como entregue, marcado_entregue"
1,Delivery - Delay,1909,16.96,1.23,"atraso, atrasado, atrasada, atrasou, demora, demorou, fora_prazo, prazo"
4,Seller/Service/Refund,1499,13.32,1.15,"reembolso, estorno, devolucao, pedi_reembolso, pedi_estorno, atendimento, sem_resposta, contato"
3,Product - Defect/Quality,1167,10.37,1.19,"defeito, defeituoso, defeituosa, quebrado, quebrada, quebrados, danificado, danificada"
5,Positive Recommendation,1127,10.01,1.29,"recomendo, gostei, otimo, excelente, bom, perfeito, chegou_antes, dentro_prazo"
2,Product - Wrong/Incomplete,1097,9.75,1.20,"produto_errado, produto_diferente, produto_incompleto, faltou, faltando, incompleto, diferente, tamanho_errado"


## 10. TF-IDF Top Terms by Score Segment

Top terms memakai `phrase_text` agar phrase token seperti `nao_recebi` tidak pecah menjadi kata generik.

In [14]:
top_terms = build_top_terms(cleaned, matching_stopwords)
top_terms[top_terms["segment"].eq("low_rating_score_1_2")].head(30)

,segment,term,score_or_weight,raw_count
60,low_rating_score_1_2,produto,0.062756,6401
61,low_rating_score_1_2,nao,0.057291,6386
62,low_rating_score_1_2,nao_recebi,0.044321,1993
63,low_rating_score_1_2,nao_recebi produto,0.029435,1028
64,low_rating_score_1_2,comprei,0.026199,1900
65,low_rating_score_1_2,entregue,0.024649,1432
66,low_rating_score_1_2,entrega,0.024567,1561
67,low_rating_score_1_2,recebi,0.023124,1510
68,low_rating_score_1_2,veio,0.022036,1367
69,low_rating_score_1_2,ainda,0.019143,1001


## 11. NMF Topic Modeling Preferred

In [15]:
nmf_topics = build_nmf_topics(cleaned, topic_stopwords)
nmf_topics

,topic_id,top_terms,manual_topic_label,review_count
2,2,"nota, defeito, outro, produto_errado, recomendo, nem, fiscal, nota fiscal, site, sem, pois, fiz",Product - Wrong/Incomplete,6619
0,0,"nao_recebi, nao_recebi nao_recebi, nao_recebi mercadoria, avaliar, nao_recebi nem, mercadoria, posso, pois nao_recebi, data, prazo nao_r...",Delivery - Not Received,1691
5,5,"pessima, qualidade, pessima qualidade, gostei, ruim, recomendo, material, baixa, baixa qualidade, foto, achei, ma",Positive Recommendation,982
3,3,"prazo, passou, passou prazo, prazo nao_recebi, antes, antes prazo, nao_recebi prazo, longo, prazo porem, nao_chegou prazo, prazo longo, ...",Delivery - Not Received,788
1,1,"nao_chegou, mercadoria, nao_chegou prazo, esperando, mes, data, prazo nao_chegou, faz, previsao, correios, consta, nao_chegou residencia",Delivery - Delay,459
4,4,"aguardando, aguardando retorno, retorno, nao_recebi aguardando, aguardando resposta, resposta, chegar, aguardando contato, solucao, envi...",Delivery - Not Received,439


## 12. LDA Topic Modeling Exploratory Comparison

In [16]:
lda_topics = build_lda_topics(cleaned, topic_stopwords)
lda_topics

,topic_id,top_terms,manual_topic_label,review_count
1,1,"nao_chegou, nao_recebi, prazo, nem, mercadoria, paguei, correios, demora, pra, frete, mes, nenhuma",Delivery - Delay,2274
2,2,"nao_recebi, aguardando, prazo, contato, problema, defeito, falta, nunca, vez, troca, compro, cliente",Delivery - Not Received,2108
5,5,"nota, outro, fiscal, nota fiscal, receber, unidades, cor, entregaram, q, contato, valor, correios",Delivery - Delay,1780
3,3,"qualidade, dinheiro, pessima, ruim, relogio, devolucao, sem, volta, recomendo, original, dinheiro volta, pessima qualidade",Product - Defect/Quality,1669
0,0,"nao_recebi, bom, aguardo, data, pois, presente, sobre, kit, site, retorno, porem, posso",Delivery - Not Received,1636
4,4,"diferente, produto_errado, foto, gostei, pedi, site, boa, outra, bem, anuncio, totalmente, entrar",Product - Wrong/Incomplete,1511


## 13. Sample Low-Rating Reviews per Theme

In [17]:
theme_cols = [col for col in theme_tags.columns if col.startswith("theme_") and col not in ["theme_other", "theme_count"]]
samples = []
for theme in theme_cols:
    matched_ids = theme_tags.loc[(theme_tags[theme]) & (theme_tags["review_score"] <= 2), "review_id"].head(3)
    sample_rows = cleaned[cleaned["review_id"].isin(matched_ids)][["review_id", "review_score", "clean_text"]].copy()
    sample_rows.insert(0, "theme", theme)
    samples.append(sample_rows)

theme_samples = pd.concat(samples, ignore_index=True)
theme_samples

,theme,review_id,review_score,clean_text
0,theme_delivery_not_received,373cbeecea8286a2b66c97b1b157ec46,1,não chegou meu produto péssimo
1,theme_delivery_not_received,58044bca115705a48fe0e00a21390c54,1,"sempre compro pela internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. no stark o prazo máximo já se ..."
2,theme_delivery_not_received,6d06808638ec0701bccd70bc8d462c28,1,"o produto não chegou no prazo estipulado e causou transtorno, pq programei a viagem de férias do meu filho, baseado no prazo. moro na ba..."
3,theme_delivery_delay,58044bca115705a48fe0e00a21390c54,1,"sempre compro pela internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. no stark o prazo máximo já se ..."
4,theme_delivery_delay,6d06808638ec0701bccd70bc8d462c28,1,"o produto não chegou no prazo estipulado e causou transtorno, pq programei a viagem de férias do meu filho, baseado no prazo. moro na ba..."
5,theme_delivery_delay,5c37d7ba6ef2f031c34bb4fda3454018,2,demorou de mais pra entrega
6,theme_product_wrong_incomplete,e233e51d11511bf30e568c76360ace52,1,recebi somente 1 controle midea split estilo. faltou controle remoto para ar condicionado consul
7,theme_product_wrong_incomplete,c40a6b6e0181e5ec0d12cbc2e12c49d3,1,falta de produto e quebra faltou 1 produto e os que recebi 1 veio quebrado
8,theme_product_wrong_incomplete,c239d18d7e310e477d5e7b76b362db1d,1,produto foi entregue com umas das alças com problemas está faltando 1 pino de fixação
9,theme_product_defect_quality,373cbeecea8286a2b66c97b1b157ec46,1,não chegou meu produto péssimo


## 14. Save Outputs

In [18]:
audit = build_audit(reviews, cleaned)
write_outputs(cleaned, theme_tags, theme_summary, top_terms, nmf_topics, lda_topics, audit)

print(json.dumps(audit, indent=2))
print(f"Outputs saved to: {OUTPUT_DIR}")

{
  "total_reviews": 99224,
  "reviews_with_title": 11566,
  "reviews_with_message": 40950,
  "reviews_with_any_text": 42454,
  "low_rating_reviews": 14575,
  "low_rating_reviews_with_text": 10978,
  "percentage_reviews_with_text": 42.79,
  "percentage_low_rating_with_text": 75.32,
  "median_token_count_all_text": 9.0,
  "median_token_count_low_rating_text": 16.0
}
Outputs saved to: C:\Kuliah ITS Farhan\Semester 4\Admin_Lab\MCI\FP_MCI_Farhan\farhan_fp_mci_customer_experience\data\processed


## 15. Final Insight Summary and Limitations

- Low-rating reviews with text are more common than text in reviews overall, which makes NLP useful for root-cause analysis.
- Delivery not received and delivery delay are important complaint groups, but not all low-rating text can be captured by explicit keyword rules.
- Phrase-aware preprocessing improves interpretability by preserving complaint expressions such as `nao_recebi`, `nao_chegou`, and `produto_errado`.
- NMF is preferred over LDA for topic interpretation because its topics are easier to inspect and label, while LDA remains exploratory.
- This analysis is descriptive and not causal proof. It must not be used as a feature source for low-rating prediction models.